In [1]:
import pandas as pd
from pathlib import Path

audio_path = Path("../data/audio")

set_a_labels = pd.read_csv(audio_path / "set_a.csv")
set_b_labels = pd.read_csv(audio_path / "set_b.csv")

print("SET A shape:", set_a_labels.shape)
print("SET B shape:", set_b_labels.shape)

print("\nSET A columns:")
print(set_a_labels.columns.tolist())

print("\nSET B columns:")
print(set_b_labels.columns.tolist())

print("\nSET A sample:")
display(set_a_labels.head())

print("\nSET B sample:")
display(set_b_labels.head())

SET A shape: (176, 4)
SET B shape: (656, 4)

SET A columns:
['dataset', 'fname', 'label', 'sublabel']

SET B columns:
['dataset', 'fname', 'label', 'sublabel']

SET A sample:


,dataset,fname,label,sublabel
0,a,set_a/artifact__201012172012.wav,artifact,NaN
1,a,set_a/artifact__201105040918.wav,artifact,NaN
2,a,set_a/artifact__201105041959.wav,artifact,NaN
3,a,set_a/artifact__201105051017.wav,artifact,NaN
4,a,set_a/artifact__201105060108.wav,artifact,NaN



SET B sample:


,dataset,fname,label,sublabel
0,b,set_b/Btraining_extrastole_127_1306764300147_C...,extrastole,NaN
1,b,set_b/Btraining_extrastole_128_1306344005749_A...,extrastole,NaN
2,b,set_b/Btraining_extrastole_130_1306347376079_D...,extrastole,NaN
3,b,set_b/Btraining_extrastole_134_1306428161797_C...,extrastole,NaN
4,b,set_b/Btraining_extrastole_138_1306762146980_B...,extrastole,NaN


In [2]:
print("SET A columns:")
print(set_a_labels.columns.tolist())

print("\nSET B columns:")
print(set_b_labels.columns.tolist())

print("\nSET A label counts:")
print(set_a_labels.iloc[:, 3].value_counts(dropna=False))

print("\nSET B label counts:")
print(set_b_labels.iloc[:, 3].value_counts(dropna=False))

SET A columns:
['dataset', 'fname', 'label', 'sublabel']

SET B columns:
['dataset', 'fname', 'label', 'sublabel']

SET A label counts:
sublabel
NaN    176
Name: count, dtype: int64

SET B label counts:
sublabel
NaN            507
noisynormal    120
noisymurmur     29
Name: count, dtype: int64


In [3]:
print("SET A actual label counts:")
print(set_a_labels["label"].value_counts(dropna=False))

print("\nSET B actual label counts:")
print(set_b_labels["label"].value_counts(dropna=False))

SET A actual label counts:
label
NaN         52
artifact    40
murmur      34
normal      31
extrahls    19
Name: count, dtype: int64

SET B actual label counts:
label
normal        320
NaN           195
murmur         95
extrastole     46
Name: count, dtype: int64


In [4]:
print("SET A unique labels:")
print(set_a_labels["label"].unique())

print("\nSET B unique labels:")
print(set_b_labels["label"].unique())

SET A unique labels:
<ArrowStringArray>
['artifact', 'extrahls', 'murmur', 'normal', nan]
Length: 5, dtype: str

SET B unique labels:
<ArrowStringArray>
['extrastole', 'murmur', 'normal', nan]
Length: 4, dtype: str


In [5]:
# Combine labels from Set A and Set B

audio_labels = pd.concat(
    [set_a_labels, set_b_labels],
    ignore_index=True
)

print("Combined dataset shape:", audio_labels.shape)

# Keep only rows with valid labels
valid_labels = [
    "normal",
    "murmur",
    "extrahls",
    "extrastole"
]

audio_labels = audio_labels[
    audio_labels["label"].isin(valid_labels)
].copy()

# Convert labels into two classes
audio_labels["target"] = audio_labels["label"].apply(
    lambda label: 0 if label == "normal" else 1
)

print("Cleaned dataset shape:", audio_labels.shape)

print("\nOriginal label counts:")
print(audio_labels["label"].value_counts())

print("\nBinary target counts:")
print(audio_labels["target"].value_counts())

print("\nTarget meaning:")
print("0 = Normal")
print("1 = Abnormal")

Combined dataset shape: (832, 4)
Cleaned dataset shape: (545, 5)

Original label counts:
label
normal        351
murmur        129
extrastole     46
extrahls       19
Name: count, dtype: int64

Binary target counts:
target
0    351
1    194
Name: count, dtype: int64

Target meaning:
0 = Normal
1 = Abnormal


In [6]:
# Check whether the audio files listed in the CSV actually exist

audio_labels["file_path"] = audio_labels["fname"].apply(
    lambda filename: audio_path / filename
)

audio_labels["file_exists"] = audio_labels["file_path"].apply(
    lambda path: path.exists()
)

print("Total labeled audio files:", len(audio_labels))
print("Files found:", audio_labels["file_exists"].sum())
print("Files missing:", (~audio_labels["file_exists"]).sum())

print("\nExample file paths:")
display(audio_labels[["fname", "file_path", "file_exists"]].head())

Total labeled audio files: 545
Files found: 84
Files missing: 461

Example file paths:


,fname,file_path,file_exists
40,set_a/extrahls__201101070953.wav,..\data\audio\set_a\extrahls__201101070953.wav,True
41,set_a/extrahls__201101091153.wav,..\data\audio\set_a\extrahls__201101091153.wav,True
42,set_a/extrahls__201101152255.wav,..\data\audio\set_a\extrahls__201101152255.wav,True
43,set_a/extrahls__201101160804.wav,..\data\audio\set_a\extrahls__201101160804.wav,True
44,set_a/extrahls__201101160808.wav,..\data\audio\set_a\extrahls__201101160808.wav,True


In [7]:
from pathlib import Path

set_a_folder = audio_path / "set_a"
set_b_folder = audio_path / "set_b"

print("Set A exists:", set_a_folder.exists())
print("Set B exists:", set_b_folder.exists())

set_a_files = list(set_a_folder.rglob("*.wav"))
set_b_files = list(set_b_folder.rglob("*.wav"))

print("\nActual WAV files in set_a:", len(set_a_files))
print("Actual WAV files in set_b:", len(set_b_files))

print("\nExample Set A files:")
for file in set_a_files[:5]:
    print(file)

print("\nExample Set B files:")
for file in set_b_files[:5]:
    print(file)

Set A exists: True
Set B exists: True

Actual WAV files in set_a: 176
Actual WAV files in set_b: 656

Example Set A files:
..\data\audio\set_a\artifact__201012172012.wav
..\data\audio\set_a\artifact__201105040918.wav
..\data\audio\set_a\artifact__201105041959.wav
..\data\audio\set_a\artifact__201105051017.wav
..\data\audio\set_a\artifact__201105060108.wav

Example Set B files:
..\data\audio\set_b\Bunlabelledtest__101_1305030823364_A.wav
..\data\audio\set_b\Bunlabelledtest__101_1305030823364_D.wav
..\data\audio\set_b\Bunlabelledtest__101_1305030823364_F.wav
..\data\audio\set_b\Bunlabelledtest__103_1305031931979_A.wav
..\data\audio\set_b\Bunlabelledtest__103_1305031931979_C.wav


In [9]:
# Create a mapping using the complete relative path

audio_file_map = {
    str(file.relative_to(audio_path)).replace("\\", "/"): file
    for file in all_audio_files
}

audio_labels["file_path"] = audio_labels["fname"].apply(
    lambda filename: audio_file_map.get(filename)
)

audio_labels["file_exists"] = audio_labels["file_path"].notna()

print("Total labeled records:", len(audio_labels))
print("Files found:", audio_labels["file_exists"].sum())
print("Files missing:", (~audio_labels["file_exists"]).sum())

audio_labels.head()

Total labeled records: 545
Files found: 84
Files missing: 461


,dataset,fname,label,sublabel,target,file_path,file_exists
40,a,set_a/extrahls__201101070953.wav,extrahls,NaN,1,..\data\audio\set_a\extrahls__201101070953.wav,True
41,a,set_a/extrahls__201101091153.wav,extrahls,NaN,1,..\data\audio\set_a\extrahls__201101091153.wav,True
42,a,set_a/extrahls__201101152255.wav,extrahls,NaN,1,..\data\audio\set_a\extrahls__201101152255.wav,True
43,a,set_a/extrahls__201101160804.wav,extrahls,NaN,1,..\data\audio\set_a\extrahls__201101160804.wav,True
44,a,set_a/extrahls__201101160808.wav,extrahls,NaN,1,..\data\audio\set_a\extrahls__201101160808.wav,True


In [10]:
print("First 15 Set B CSV filenames:")

for filename in set_b_labels["fname"].head(15):
    print(filename)

First 15 Set B CSV filenames:
set_b/Btraining_extrastole_127_1306764300147_C2.wav
set_b/Btraining_extrastole_128_1306344005749_A.wav
set_b/Btraining_extrastole_130_1306347376079_D.wav
set_b/Btraining_extrastole_134_1306428161797_C1.wav
set_b/Btraining_extrastole_138_1306762146980_B.wav
set_b/Btraining_extrastole_140_1306519735121_D.wav
set_b/Btraining_extrastole_144_1306522408528_B.wav
set_b/Btraining_extrastole_144_1306522408528_B1.wav
set_b/Btraining_extrastole_148_1306768801551_B.wav
set_b/Btraining_extrastole_151_1306779785624_B.wav
set_b/Btraining_extrastole_153_1306848820671_C.wav
set_b/Btraining_extrastole_154_1306935608852_D2.wav
set_b/Btraining_extrastole_163_1307104470471_C.wav
set_b/Btraining_extrastole_179_1307990076841_C.wav
set_b/Btraining_extrastole_184_1308073010307_A.wav


In [11]:
# Compare Set B CSV filenames with actual Set B files

set_b_file_map = {
    file.name: file
    for file in set_b_files
}

print("CSV filename:", set_b_labels.iloc[0]["fname"])
print("CSV basename:", Path(set_b_labels.iloc[0]["fname"]).name)
print(
    "Matching actual file:",
    set_b_file_map.get(Path(set_b_labels.iloc[0]["fname"]).name)
)

CSV filename: set_b/Btraining_extrastole_127_1306764300147_C2.wav
CSV basename: Btraining_extrastole_127_1306764300147_C2.wav
Matching actual file: None


In [12]:
from collections import Counter

print("Actual Set B filename prefixes:")

prefixes = Counter(
    file.name.split("__")[0]
    for file in set_b_files
)

print(prefixes)

Actual Set B filename prefixes:
Counter({'normal': 200, 'Bunlabelledtest': 195, 'murmur': 66, 'extrastole': 46, 'murmur_noisymurmur_135_1306428972976_A.wav': 1, 'murmur_noisymurmur_135_1306428972976_B.wav': 1, 'murmur_noisymurmur_135_1306428972976_C.wav': 1, 'murmur_noisymurmur_156_1306936373241_A.wav': 1, 'murmur_noisymurmur_156_1306936373241_B1.wav': 1, 'murmur_noisymurmur_160_1307100683334_D.wav': 1, 'murmur_noisymurmur_161_1307101199321_B.wav': 1, 'murmur_noisymurmur_161_1307101199321_C.wav': 1, 'murmur_noisymurmur_162_1307101835989_B_1.wav': 1, 'murmur_noisymurmur_162_1307101835989_D.wav': 1, 'murmur_noisymurmur_164_1307106095995_C1.wav': 1, 'murmur_noisymurmur_165_1307109069581_A.wav': 1, 'murmur_noisymurmur_165_1307109069581_C1.wav': 1, 'murmur_noisymurmur_165_1307109069581_D.wav': 1, 'murmur_noisymurmur_171_1307971016233_D.wav': 1, 'murmur_noisymurmur_171_1307971016233_F.wav': 1, 'murmur_noisymurmur_185_1308073325396_D.wav': 1, 'murmur_noisymurmur_200_1308144251434_D.wav': 1, '

In [13]:
from pathlib import Path
import pandas as pd

audio_records = []

for file in all_audio_files:
    filename = file.name.lower()

    if filename.startswith("normal"):
        label = "normal"
        target = 0

    elif filename.startswith("murmur"):
        label = "murmur"
        target = 1

    elif filename.startswith("extrastole"):
        label = "extrastole"
        target = 1

    elif filename.startswith("extrahls"):
        label = "extrahls"
        target = 1

    else:
        # Ignore artifact and unlabelled recordings
        continue

    audio_records.append({
        "file_path": str(file),
        "label": label,
        "target": target
    })

audio_data = pd.DataFrame(audio_records)

print("Total usable audio files:", len(audio_data))
print("\nLabel counts:")
print(audio_data["label"].value_counts())

print("\nTarget counts:")
print(audio_data["target"].value_counts())

audio_data.head()

Total usable audio files: 545

Label counts:
label
normal        351
murmur        129
extrastole     46
extrahls       19
Name: count, dtype: int64

Target counts:
target
0    351
1    194
Name: count, dtype: int64


,file_path,label,target
0,..\data\audio\set_a\extrahls__201101070953.wav,extrahls,1
1,..\data\audio\set_a\extrahls__201101091153.wav,extrahls,1
2,..\data\audio\set_a\extrahls__201101152255.wav,extrahls,1
3,..\data\audio\set_a\extrahls__201101160804.wav,extrahls,1
4,..\data\audio\set_a\extrahls__201101160808.wav,extrahls,1


In [15]:
import librosa
import numpy as np

print("Librosa version:", librosa.__version__)

Librosa version: 1.0.0


In [16]:
import librosa
import numpy as np
import pandas as pd

def extract_features(file_path):
    audio, sample_rate = librosa.load(
        file_path,
        sr=22050,
        mono=True
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sample_rate,
        n_mfcc=13
    )

    features = np.concatenate([
        np.mean(mfcc, axis=1),
        np.std(mfcc, axis=1)
    ])

    return features


feature_rows = []
valid_paths = []
failed_files = []

for index, file_path in enumerate(audio_data["file_path"]):
    try:
        features = extract_features(file_path)
        feature_rows.append(features)
        valid_paths.append(file_path)

    except Exception as error:
        failed_files.append((file_path, str(error)))

    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1}/{len(audio_data)} files")


X_audio = np.array(feature_rows)

audio_features = pd.DataFrame(
    X_audio,
    columns=[f"mfcc_mean_{i+1}" for i in range(13)]
            + [f"mfcc_std_{i+1}" for i in range(13)]
)

audio_features["file_path"] = valid_paths

audio_features = audio_features.merge(
    audio_data[["file_path", "label", "target"]],
    on="file_path",
    how="left"
)

print("Feature shape:", audio_features.shape)
print("Failed files:", len(failed_files))
audio_features.head()

Processed 50/545 files
Processed 100/545 files
Processed 150/545 files
Processed 200/545 files
Processed 250/545 files
Processed 300/545 files
Processed 350/545 files
Processed 400/545 files
Processed 450/545 files
Processed 500/545 files
Feature shape: (545, 29)
Failed files: 0


,mfcc_mean_1,mfcc_mean_2,mfcc_mean_3,mfcc_mean_4,mfcc_mean_5,mfcc_mean_6,mfcc_mean_7,mfcc_mean_8,mfcc_mean_9,mfcc_mean_10,...,mfcc_std_7,mfcc_std_8,mfcc_std_9,mfcc_std_10,mfcc_std_11,mfcc_std_12,mfcc_std_13,file_path,label,target
0,-496.182404,62.221668,9.681172,22.574318,9.242278,13.187013,5.588273,7.906901,2.661797,4.181629,...,7.649493,5.990260,4.852529,4.300045,3.585046,3.058797,3.196588,..\data\audio\set_a\extrahls__201101070953.wav,extrahls,1
1,-454.656342,99.458130,17.842161,39.309662,14.872849,22.087341,7.965159,9.321386,0.298609,2.134581,...,5.759756,4.573285,4.481387,4.047940,4.151454,4.288453,4.307868,..\data\audio\set_a\extrahls__201101091153.wav,extrahls,1
2,-671.620361,35.237072,-0.051869,21.526402,3.983726,9.687203,1.883760,4.830184,-0.493477,2.354839,...,6.837013,5.195774,4.431772,3.919528,3.784570,3.524368,3.414623,..\data\audio\set_a\extrahls__201101152255.wav,extrahls,1
3,-452.145416,122.472076,9.030285,24.747097,14.514076,29.144680,10.943706,6.807246,0.550476,8.726333,...,6.511080,5.387522,5.322914,5.666871,5.271420,4.350839,4.437339,..\data\audio\set_a\extrahls__201101160804.wav,extrahls,1
4,-433.947876,95.640289,7.806315,32.774254,14.667113,22.765234,8.611640,8.442687,1.056993,3.992747,...,7.846646,8.601686,8.431993,6.022888,4.975855,4.952487,4.629673,..\data\audio\set_a\extrahls__201101160808.wav,extrahls,1


In [17]:
from sklearn.model_selection import train_test_split

# Select only the numerical audio features
feature_columns = [
    column
    for column in audio_features.columns
    if column.startswith("mfcc_")
]

X = audio_features[feature_columns]
y = audio_features["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Number of features:", X_train.shape[1])
print("Training class distribution:")
print(y_train.value_counts())

Training samples: 436
Testing samples: 109
Number of features: 26
Training class distribution:
target
0    281
1    155
Name: count, dtype: int64


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

audio_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

audio_model.fit(X_train, y_train)

print("Audio model training completed successfully.")

Audio model training completed successfully.


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred = audio_model.predict(X_test)
y_prob = audio_model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Abnormal"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.6972
Precision: 0.575
Recall: 0.5897
F1 Score: 0.5823
ROC-AUC: 0.7048

Classification Report:
              precision    recall  f1-score   support

      Normal       0.77      0.76      0.76        70
    Abnormal       0.57      0.59      0.58        39

    accuracy                           0.70       109
   macro avg       0.67      0.67      0.67       109
weighted avg       0.70      0.70      0.70       109

Confusion Matrix:
[[53 17]
 [16 23]]


In [20]:
from pathlib import Path
import joblib

models_path = Path("../models")
models_path.mkdir(exist_ok=True)

audio_model_path = models_path / "audio_heartbeat_model.pkl"

joblib.dump(audio_model, audio_model_path)

print("Audio model saved successfully.")
print("Saved at:", audio_model_path)

Audio model saved successfully.
Saved at: ..\models\audio_heartbeat_model.pkl


In [21]:
import joblib
from pathlib import Path

saved_audio_model = joblib.load(
    Path("../models/audio_heartbeat_model.pkl")
)

print("Saved audio model loaded successfully.")
print(saved_audio_model)

Saved audio model loaded successfully.
Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])


In [22]:
import librosa
import numpy as np
import pandas as pd

# Select one audio file from the dataset
sample_file = audio_features["file_path"].iloc[0]

# Extract its features
sample_features = extract_features(sample_file)

# Convert features into a DataFrame with the same column names
sample_input = pd.DataFrame(
    [sample_features],
    columns=feature_columns
)

# Make prediction
prediction = saved_audio_model.predict(sample_input)[0]
probability = saved_audio_model.predict_proba(sample_input)[0][1]

print("Audio file:", sample_file)
print("Prediction:", "Abnormal" if prediction == 1 else "Normal")
print("Abnormal probability:", round(probability * 100, 2), "%")

Audio file: ..\data\audio\set_a\extrahls__201101070953.wav
Prediction: Abnormal
Abnormal probability: 59.98 %
